## Step 0 — Setup and Load

Before any transformation, the data must be in a known, correct state.

Two critical things we do here:
1. Sort by ['Ticker', 'Date'] — rolling windows depend on row order.
   Wrong order = silently corrupted features. No error will warn you.
2. Reset index — after sorting the old index is meaningless.

We also create the ../data/processed/ directory if it does not exist.


In [1]:
# ============================================================
# CELL 1 - Imports and Directory Setup
# ============================================================
# Import all libraries needed for the full notebook upfront
# Create output directory if it does not already exist
# ============================================================

import pandas as pd
import numpy as np
import ta
import os
import warnings
warnings.filterwarnings('ignore')

os.makedirs('../data/processed', exist_ok=True)

print('Libraries loaded successfully')
print(f'  pandas : {pd.__version__}')
print(f'  numpy  : {np.__version__}')
print(f'  ta     : installed OK')
print(f'  Output : ../data/processed/ ready')

Libraries loaded successfully
  pandas : 3.0.3
  numpy  : 2.4.6
  ta     : installed OK
  Output : ../data/processed/ ready


In [2]:
# ============================================================
# CELL 2 - Load Raw Data
# ============================================================
# Load the raw CSV saved by notebook 01
# Sort by Ticker then Date — mandatory before any rolling/shift
# Reset index for a clean 0-to-N integer index
# ============================================================

df = pd.read_csv('../data/raw/all_stocks_raw.csv', parse_dates=['Date'])

df = df.sort_values(['Ticker', 'Date']).reset_index(drop=True)

print('Raw data loaded and sorted')
print('=' * 45)
print(f'  Rows         : {len(df):,}')
print(f'  Columns      : {list(df.columns)}')
print(f'  Tickers      : {df["Ticker"].nunique()} stocks')
print(f'  Date range   : {df["Date"].min().date()} to {df["Date"].max().date()}')
print(f'  Missing vals : {df.isnull().sum().sum()}')
print('=' * 45)
print(df.head(3).to_string())

Raw data loaded and sorted
  Rows         : 30,180
  Columns      : ['Date', 'Close', 'High', 'Low', 'Open', 'Volume', 'Ticker']
  Tickers      : 20 stocks
  Date range   : 2019-01-02 to 2024-12-30
  Missing vals : 0
        Date      Close       High        Low       Open     Volume Ticker
0 2019-01-02  37.469204  37.689864  36.593688  36.750285  148158800   AAPL
1 2019-01-03  33.736988  34.574540  33.691907  34.161694  365248800   AAPL
2 2019-01-04  35.177197  35.246006  34.118988  34.292192  234428400   AAPL


---------------------------------

## Step 1 — Price-Based Features

Raw Close price cannot be used as a model feature.
$180 AAPL and $180 NVDA are the same number but completely different situations.

Returns (percentage changes) solve this.
A +2% move means the same thing whether the stock is at $50 or $500.

Features built here:

| Feature        | Formula                  | What it captures          |
|----------------|--------------------------|---------------------------|
| return_1d      | Close pct_change(1)      | Yesterday's momentum      |
| return_5d      | Close pct_change(5)      | Weekly trend              |
| return_10d     | Close pct_change(10)     | Bi-weekly trend           |
| return_20d     | Close pct_change(20)     | Monthly trend             |
| price_vs_ma20  | Close / MA20 - 1         | Distance from monthly avg |
| price_vs_ma50  | Close / MA50 - 1         | Distance from quarterly avg|

Why ratio for MA and not just above/below boolean?
Being 0.1% above MA20 is very different from being 8% above it.
The ratio tells the model HOW STRETCHED the price is — that is what matters.

All operations use groupby('Ticker') so AAPL's window never touches MSFT.


In [3]:
# ============================================================
# CELL 3 - Return Features (1d, 5d, 10d, 20d)
# ============================================================
# pct_change(n) = (today - n_days_ago) / n_days_ago
# groupby ensures each ticker's window is self-contained
# transform returns result aligned to the original dataframe index
# ============================================================

df['return_1d']  = df.groupby('Ticker')['Close'].transform(
    lambda x: x.pct_change(1)
)
df['return_5d']  = df.groupby('Ticker')['Close'].transform(
    lambda x: x.pct_change(5)
)
df['return_10d'] = df.groupby('Ticker')['Close'].transform(
    lambda x: x.pct_change(10)
)
df['return_20d'] = df.groupby('Ticker')['Close'].transform(
    lambda x: x.pct_change(20)
)

print('Return features created')
print('=' * 45)
print(f'  return_1d  : {df["return_1d"].isna().sum():>4} NaN rows (expected {1*20})')
print(f'  return_5d  : {df["return_5d"].isna().sum():>4} NaN rows (expected {5*20})')
print(f'  return_10d : {df["return_10d"].isna().sum():>4} NaN rows (expected {10*20})')
print(f'  return_20d : {df["return_20d"].isna().sum():>4} NaN rows (expected {20*20})')

Return features created
  return_1d  :   20 NaN rows (expected 20)
  return_5d  :  100 NaN rows (expected 100)
  return_10d :  200 NaN rows (expected 200)
  return_20d :  400 NaN rows (expected 400)


In [4]:
# ============================================================
# CELL 4 - Price vs Moving Average Features
# ============================================================
# MA20 = 20-day rolling mean of Close price per ticker
# MA50 = 50-day rolling mean of Close price per ticker
# price_vs_ma = Close / MA - 1
#   Positive = price is above average (bullish stretch)
#   Negative = price is below average (bearish stretch)
#   Zero     = price sitting exactly on the average
# ============================================================

ma20 = df.groupby('Ticker')['Close'].transform(
    lambda x: x.rolling(window=20).mean()
)
ma50 = df.groupby('Ticker')['Close'].transform(
    lambda x: x.rolling(window=50).mean()
)

df['price_vs_ma20'] = (df['Close'] / ma20) - 1
df['price_vs_ma50'] = (df['Close'] / ma50) - 1

print('Moving average features created')
print('=' * 45)
print(f'  price_vs_ma20 : {df["price_vs_ma20"].isna().sum():>4} NaN rows (expected {19*20})')
print(f'  price_vs_ma50 : {df["price_vs_ma50"].isna().sum():>4} NaN rows (expected {49*20})')
print('=' * 45)
print('\nSample values for AAPL (rows 50-53):')
cols = ['Date', 'Close', 'price_vs_ma20', 'price_vs_ma50']
print(df[df['Ticker'] == 'AAPL'][cols].iloc[50:54].to_string(index=False))

Moving average features created
  price_vs_ma20 :  380 NaN rows (expected 380)
  price_vs_ma50 :  980 NaN rows (expected 980)

Sample values for AAPL (rows 50-53):
      Date     Close  price_vs_ma20  price_vs_ma50
2019-03-15 44.349537       0.060018       0.123474
2019-03-18 44.802273       0.065499       0.128615
2019-03-19 44.447227       0.052403       0.114466
2019-03-20 44.835636       0.056791       0.118743


In [5]:
# ============================================================
# CELL 5 - Step 1 Validation
# ============================================================
# Confirm all 6 price features exist and have sensible values
# Check value ranges - returns should not be extreme outliers
# ============================================================

price_features = ['return_1d', 'return_5d', 'return_10d',
                  'return_20d', 'price_vs_ma20', 'price_vs_ma50']

print('Step 1 Complete - Price Feature Summary')
print('=' * 55)
print(f'{"Feature":<18} {"Mean":>8} {"Std":>8} {"Min":>8} {"Max":>8}')
print('-' * 55)
for col in price_features:
    series = df[col].dropna()
    print(f'{col:<18} {series.mean():>8.4f} {series.std():>8.4f} '
          f'{series.min():>8.4f} {series.max():>8.4f}')

print('=' * 55)
print(f'\nTotal features so far : {len(price_features)}')
print(f'Dataframe shape       : {df.shape}')

Step 1 Complete - Price Feature Summary
Feature                Mean      Std      Min      Max
-------------------------------------------------------
return_1d            0.0011   0.0224  -0.3512   0.2604
return_5d            0.0054   0.0484  -0.4309   0.5648
return_10d           0.0107   0.0682  -0.5181   0.6588
return_20d           0.0215   0.0985  -0.6063   1.0630
price_vs_ma20        0.0085   0.0534  -0.4632   0.5502
price_vs_ma50        0.0216   0.0863  -0.4629   0.9526

Total features so far : 6
Dataframe shape       : (30180, 13)


-----------------------------------------------------

## Step 2 — Technical Indicators

Price and volume alone do not tell the full story.
Technical indicators compress complex price behaviour into single numbers
that traders and algorithms have used for decades.

We use the `ta` library — industry standard for Python financial analysis.
All indicators are computed per Ticker using groupby to prevent data leakage.

Indicators built here:

| Feature       | Type       | What it captures                        |
|---------------|------------|-----------------------------------------|
| rsi_14        | Momentum   | Overbought / oversold conditions        |
| macd          | Trend      | Difference between fast and slow EMA    |
| macd_signal   | Trend      | 9-day EMA of MACD line                  |
| macd_hist     | Trend      | MACD minus Signal — actual trade signal |
| bb_width      | Volatility | How wide the Bollinger Bands are        |
| atr_14        | Volatility | Average size of daily price swings      |

Why these four indicators specifically?
- RSI   : catches momentum reversals — overbought stocks tend to fall
- MACD  : catches trend changes — when fast EMA crosses slow EMA
- BB    : catches volatility expansions — wide bands = big moves coming
- ATR   : measures raw volatility — normalised by price for comparability

In [6]:
# ============================================================
# CELL 6 - RSI (Relative Strength Index)
# ============================================================
# RSI measures momentum on a 0-100 scale
#   Above 70 = overbought = stock may be due for a pullback
#   Below 30 = oversold   = stock may be due for a bounce
#   50       = neutral
# Standard period is 14 days - used by traders worldwide
# ta.momentum.RSIIndicator expects a Series, returns a Series
# ============================================================

def compute_rsi(group):
    return ta.momentum.RSIIndicator(
        close=group['Close'],
        window=14
    ).rsi()

df['rsi_14'] = df.groupby('Ticker', group_keys=False).apply(compute_rsi)

print('RSI computed')
print('=' * 45)
print(f'  rsi_14 NaNs  : {df["rsi_14"].isna().sum()} (expected {14*20})')
print(f'  rsi_14 min   : {df["rsi_14"].min():.2f}  (should be > 0)')
print(f'  rsi_14 max   : {df["rsi_14"].max():.2f}  (should be < 100)')
print(f'  rsi_14 mean  : {df["rsi_14"].mean():.2f}  (should be near 50)')

RSI computed
  rsi_14 NaNs  : 260 (expected 280)
  rsi_14 min   : 9.15  (should be > 0)
  rsi_14 max   : 94.20  (should be < 100)
  rsi_14 mean  : 53.86  (should be near 50)


In [7]:
# ============================================================
# CELL 7 - MACD (Moving Average Convergence Divergence)
# ============================================================
# MACD = 12-day EMA minus 26-day EMA
# Signal = 9-day EMA of the MACD line
# Histogram = MACD minus Signal
#
# The HISTOGRAM is the most useful feature for ML:
#   Positive histogram = MACD above signal = bullish momentum
#   Negative histogram = MACD below signal = bearish momentum
#   Crosses zero       = momentum shift = potential trade signal
#
# Standard periods: fast=12, slow=26, signal=9
# These are the industry defaults used everywhere
# ============================================================

def compute_macd(group):
    macd_obj = ta.trend.MACD(
        close=group['Close'],
        window_fast=12,
        window_slow=26,
        window_sign=9
    )
    result = pd.DataFrame(index=group.index)
    result['macd']        = macd_obj.macd()
    result['macd_signal'] = macd_obj.macd_signal()
    result['macd_hist']   = macd_obj.macd_diff()
    return result

macd_df = df.groupby('Ticker', group_keys=False).apply(compute_macd)

df['macd']        = macd_df['macd']
df['macd_signal'] = macd_df['macd_signal']
df['macd_hist']   = macd_df['macd_hist']

print('MACD computed')
print('=' * 45)
print(f'  macd NaNs        : {df["macd"].isna().sum()} (expected {26*20})')
print(f'  macd_signal NaNs : {df["macd_signal"].isna().sum()} (expected {(26+9-1)*20})')
print(f'  macd_hist NaNs   : {df["macd_hist"].isna().sum()} (expected {(26+9-1)*20})')

MACD computed
  macd NaNs        : 500 (expected 520)
  macd_signal NaNs : 660 (expected 680)
  macd_hist NaNs   : 660 (expected 680)


In [8]:
# ============================================================
# CELL 8 - Bollinger Band Width
# ============================================================
# Bollinger Bands = 20-day MA ± 2 standard deviations
#   Upper band = MA20 + 2*std
#   Lower band = MA20 - 2*std
#   Width      = (Upper - Lower) / Middle
#
# We use WIDTH not the raw bands because:
#   - Width is scale-free (works across all price levels)
#   - Wide bands  = high volatility = big move expected
#   - Narrow bands = low volatility  = compression before breakout
#   - This is called a "Bollinger Squeeze" in trading

# bollinger_wband() returns percentage (0-100 scale)
# We divide by 100 to convert to ratio (0-1 scale)
# This keeps BB width consistent with all other features
# ============================================================

def compute_bb_width(group):
    bb = ta.volatility.BollingerBands(
        close=group['Close'],
        window=20,
        window_dev=2
    )
    return bb.bollinger_wband() / 100    # convert % to ratio

df['bb_width'] = df.groupby('Ticker', group_keys=False).apply(compute_bb_width)

print('Bollinger Band Width computed (corrected)')
print('=' * 45)
print(f'  bb_width NaNs : {df["bb_width"].isna().sum()} (expected 380)')
print(f'  bb_width min  : {df["bb_width"].min():.4f}')
print(f'  bb_width max  : {df["bb_width"].max():.4f}')
print(f'  bb_width mean : {df["bb_width"].mean():.4f}')

Bollinger Band Width computed (corrected)
  bb_width NaNs : 380 (expected 380)
  bb_width min  : 0.0196
  bb_width max  : 1.1154
  bb_width mean : 0.1295


In [9]:
# ============================================================
# CELL 9 - ATR (Average True Range) - Normalised
# ============================================================
# ATR measures the average size of daily price swings
# True Range = max of:
#   High - Low           (today's range)
#   |High - prev Close|  (gap up scenario)
#   |Low  - prev Close|  (gap down scenario)
# ATR = 14-day average of True Range
#
# WHY NORMALISE BY CLOSE PRICE?
#   Raw ATR for NVDA = $8  and  raw ATR for GS = $8
#   But NVDA is at $500 so $8 = 1.6% move  (small)
#   GS is at $400 so $8 = 2.0% move         (larger)
#   Dividing by Close makes ATR scale-free and comparable

# ta library fills ATR warmup rows with 0 instead of NaN
# We replace 0 with NaN so dropna() catches them in Step 7
# A real ATR can never be zero - stock always moves some amount
# ============================================================

def compute_atr(group):
    atr = ta.volatility.AverageTrueRange(
        high=group['High'],
        low=group['Low'],
        close=group['Close'],
        window=14
    ).average_true_range()
    atr_norm = atr / group['Close']
    atr_norm = atr_norm.replace(0, np.nan)   # replace fake zeros with NaN
    return atr_norm

df['atr_14'] = df.groupby('Ticker', group_keys=False).apply(compute_atr)

print('ATR (normalised) computed - corrected')
print('=' * 45)
print(f'  atr_14 NaNs : {df["atr_14"].isna().sum()} (expected ~280)')
print(f'  atr_14 min  : {df["atr_14"].min():.4f}  (should be > 0)')
print(f'  atr_14 max  : {df["atr_14"].max():.4f}')
print(f'  atr_14 mean : {df["atr_14"].mean():.4f}')

ATR (normalised) computed - corrected
  atr_14 NaNs : 260 (expected ~280)
  atr_14 min  : 0.0095  (should be > 0)
  atr_14 max  : 0.1921
  atr_14 mean : 0.0265


In [10]:
# ============================================================
# CELL 10 - Step 2 Validation
# ============================================================
# Confirm all 6 technical indicators exist
# Check value ranges are sensible
# Print dataframe shape to confirm no accidental row drops
# ============================================================

tech_features = ['rsi_14', 'macd', 'macd_signal', 'macd_hist', 'bb_width', 'atr_14']

print('Step 2 Complete - Technical Indicator Summary')
print('=' * 60)
print(f'{"Feature":<15} {"NaNs":>6} {"Mean":>8} {"Std":>8} {"Min":>8} {"Max":>8}')
print('-' * 60)
for col in tech_features:
    series = df[col].dropna()
    print(f'{col:<15} {df[col].isna().sum():>6} {series.mean():>8.4f} '
          f'{series.std():>8.4f} {series.min():>8.4f} {series.max():>8.4f}')

print('=' * 60)
print(f'\nTotal features so far : 6 price + 6 technical = 12')
print(f'Dataframe shape       : {df.shape}')

Step 2 Complete - Technical Indicator Summary
Feature           NaNs     Mean      Std      Min      Max
------------------------------------------------------------
rsi_14             260  53.8588  12.1674   9.1523  94.1980
macd               500   0.9111   4.7354 -32.6176  40.5386
macd_signal        660   0.9058   4.4243 -29.2197  34.7566
macd_hist          660  -0.0002   1.4702 -12.0971  10.2962
bb_width           380   0.1295   0.0900   0.0196   1.1154
atr_14             260   0.0265   0.0127   0.0095   0.1921

Total features so far : 6 price + 6 technical = 12
Dataframe shape       : (30180, 19)


-----------------------------------------

## Step 3 — Volume-Based Features

Price tells us WHERE the stock moved.
Volume tells us HOW MUCH CONVICTION was behind that move.

A 3% price rise on 10x normal volume is a very strong signal.
A 3% price rise on 0.5x normal volume is likely just noise.

This is why volume is one of the most powerful predictive signals in finance.
Our EDA confirmed it — high volume days have stronger next day moves.

Features built here:

| Feature             | Formula                        | What it captures                    |
|---------------------|--------------------------------|-------------------------------------|
| volume_ratio        | Volume / 20-day avg volume     | Is today unusually busy or quiet?   |
| volume_trend        | Volume pct_change over 5 days  | Is trading activity increasing?     |
| price_volume_signal | return_1d × volume_ratio       | Strong move + high volume = conviction|

Why price_volume_signal matters:
  +return × high volume  →  genuine buying pressure  →  bullish
  -return × high volume  →  genuine selling pressure →  bearish
  any return × low volume →  weak signal, likely noise

In [11]:
# ============================================================
# CELL 11 - Volume Ratio
# ============================================================
# volume_ratio = today's volume / 20-day average volume
#   > 1.0  →  more activity than usual
#   < 1.0  →  less activity than usual
#   = 2.0  →  twice the normal volume (significant event)
# groupby ensures each ticker's average is self-contained
# ============================================================

df['volume_ratio'] = df.groupby('Ticker')['Volume'].transform(
    lambda x: x / x.rolling(window=20).mean()
)

print('Volume Ratio computed')
print('=' * 45)
print(f'  volume_ratio NaNs : {df["volume_ratio"].isna().sum()} (expected {19*20})')
print(f'  volume_ratio min  : {df["volume_ratio"].min():.4f}')
print(f'  volume_ratio max  : {df["volume_ratio"].max():.4f}')
print(f'  volume_ratio mean : {df["volume_ratio"].mean():.4f} (should be near 1.0)')

Volume Ratio computed
  volume_ratio NaNs : 380 (expected 380)
  volume_ratio min  : 0.1431
  volume_ratio max  : 11.4329
  volume_ratio mean : 1.0079 (should be near 1.0)


In [12]:
# ============================================================
# CELL 12 - Volume Trend
# ============================================================
# volume_trend = 5-day percentage change in volume
# Captures whether trading activity is growing or shrinking
#   Positive →  more people entering the market for this stock
#   Negative →  traders losing interest, volume drying up
# Volume drying up before a price move = weak conviction
# Volume building before a price move = strong conviction
# ============================================================

df['volume_trend'] = df.groupby('Ticker')['Volume'].transform(
    lambda x: x.pct_change(5)
)

print('Volume Trend computed')
print('=' * 45)
print(f'  volume_trend NaNs : {df["volume_trend"].isna().sum()} (expected {5*20})')
print(f'  volume_trend min  : {df["volume_trend"].min():.4f}')
print(f'  volume_trend max  : {df["volume_trend"].max():.4f}')
print(f'  volume_trend mean : {df["volume_trend"].mean():.4f} (should be near 0)')

Volume Trend computed
  volume_trend NaNs : 100 (expected 100)
  volume_trend min  : -0.9165
  volume_trend max  : 33.8789
  volume_trend mean : 0.1190 (should be near 0)


In [13]:
# ============================================================
# CELL 13 - Price Volume Signal
# ============================================================
# price_volume_signal = return_1d × volume_ratio
# Combines price direction with volume conviction
#
# Examples:
#   return_1d = +0.03, volume_ratio = 3.0  → signal = +0.09  strong bull
#   return_1d = +0.03, volume_ratio = 0.3  → signal = +0.009 weak bull
#   return_1d = -0.03, volume_ratio = 3.0  → signal = -0.09  strong bear
#   return_1d = -0.03, volume_ratio = 0.3  → signal = -0.009 weak bear
#
# No groupby needed here — both columns already computed per ticker
# Simple element-wise multiplication
# ============================================================

df['price_volume_signal'] = df['return_1d'] * df['volume_ratio']

print('Price Volume Signal computed')
print('=' * 45)
print(f'  price_volume_signal NaNs : {df["price_volume_signal"].isna().sum()}')
print(f'  price_volume_signal min  : {df["price_volume_signal"].min():.4f}')
print(f'  price_volume_signal max  : {df["price_volume_signal"].max():.4f}')
print(f'  price_volume_signal mean : {df["price_volume_signal"].mean():.4f} (should be near 0)')

Price Volume Signal computed
  price_volume_signal NaNs : 380
  price_volume_signal min  : -4.0149
  price_volume_signal max  : 1.8364
  price_volume_signal mean : 0.0008 (should be near 0)


In [14]:
# ============================================================
# CELL 14 - Step 3 Validation
# ============================================================
# Confirm all 3 volume features exist and have sensible values
# Verify dataframe shape — should be 22 columns now
# ============================================================

volume_features = ['volume_ratio', 'volume_trend', 'price_volume_signal']

print('Step 3 Complete - Volume Feature Summary')
print('=' * 60)
print(f'{"Feature":<22} {"NaNs":>6} {"Mean":>8} {"Std":>8} {"Min":>8} {"Max":>8}')
print('-' * 60)
for col in volume_features:
    series = df[col].dropna()
    print(f'{col:<22} {df[col].isna().sum():>6} {series.mean():>8.4f} '
          f'{series.std():>8.4f} {series.min():>8.4f} {series.max():>8.4f}')

print('=' * 60)
print(f'\nTotal features so far : 6 price + 6 technical + 3 volume = 15')
print(f'Dataframe shape       : {df.shape}')

Step 3 Complete - Volume Feature Summary
Feature                  NaNs     Mean      Std      Min      Max
------------------------------------------------------------
volume_ratio              380   1.0079   0.4333   0.1431  11.4329
volume_trend              100   0.1190   0.6801  -0.9165  33.8789
price_volume_signal       380   0.0008   0.0502  -4.0149   1.8364

Total features so far : 6 price + 6 technical + 3 volume = 15
Dataframe shape       : (30180, 22)


In [15]:
# ============================================================
# CELL - Investigate Volume Trend Outlier
# ============================================================
# Find which stock and date caused the 33x volume spike
# This helps us decide if it is real data or corruption
# ============================================================

idx = df['volume_trend'].idxmax()
row = df.loc[idx, ['Date', 'Ticker', 'Volume', 'volume_trend', 'volume_ratio']]
print('Largest volume_trend spike:')
print(row.to_string())

# Also check top 5
print('\nTop 5 volume trend spikes:')
top5 = df.nlargest(5, 'volume_trend')[['Date', 'Ticker', 'Volume', 'volume_trend', 'volume_ratio']]
print(top5.to_string(index=False))

Largest volume_trend spike:
Date            2022-04-20 00:00:00
Ticker                         NFLX
Volume                   1333875000
volume_trend              33.878932
volume_ratio              11.432927

Top 5 volume trend spikes:
      Date Ticker     Volume  volume_trend  volume_ratio
2022-04-20   NFLX 1333875000     33.878932     11.432927
2024-05-30    CRM   66763300     19.175057      8.460743
2022-04-21   NFLX  535016000     15.558836      3.779539
2022-01-21   NFLX  589043000     12.162678      8.686605
2022-09-15   ADBE   27840200     11.227776      7.327583


In [16]:
# ============================================================
# CELL - Investigate Price Volume Signal Outlier
# ============================================================

idx = df['price_volume_signal'].idxmin()
row = df.loc[idx, ['Date', 'Ticker', 'return_1d', 'volume_ratio', 'price_volume_signal']]
print('Largest negative price_volume_signal:')
print(row.to_string())

Largest negative price_volume_signal:
Date                   2022-04-20 00:00:00
Ticker                                NFLX
return_1d                        -0.351166
volume_ratio                     11.432927
price_volume_signal              -4.014856


----------------------------------------

## Step 4 — Market-Wide Feature

All 15 features built so far are computed per stock in isolation.
They capture what THAT stock is doing — but not what the MARKET is doing.

Our EDA showed that in 2022, ALL 20 stocks fell together due to Fed rate hikes.
In March 2020, ALL 20 stocks crashed together due to COVID.
No per-stock indicator can capture this — a stock's RSI only sees its own price.

We need one feature that tells the model:
  "Today the entire market is selling off" or
  "Today the entire market is rallying"

Solution: compute the average return_1d across all 20 stocks for each date.
This is a simple proxy for overall market sentiment on that day.

  market_return_1d > 0  →  broad market up    →  tailwind for all stocks
  market_return_1d < 0  →  broad market down  →  headwind for all stocks
  market_return_1d extreme negative → systemic event (COVID, rate hike panic)

Important: this is computed ACROSS tickers per date, not per ticker.
No groupby('Ticker') needed here — we deliberately want the cross-stock average.
No data leakage — we use today's market return as a feature, not future returns.

In [17]:
# ============================================================
# CELL 15 - Market-Wide Feature
# ============================================================
# Compute average return_1d across all 20 stocks per date
# This captures broad market sentiment on each trading day
# Steps:
#   1. Group by Date, take mean of return_1d across all tickers
#   2. Map this daily average back to every row in df
#   3. Each row now knows what the whole market did that day
# ============================================================

# Step 1 - compute daily market average return
market_daily = df.groupby('Date')['return_1d'].mean()

# Step 2 - map back to every row using the Date column
df['market_return_1d'] = df['Date'].map(market_daily)

print('Market-Wide Feature computed')
print('=' * 45)
print(f'  market_return_1d NaNs : {df["market_return_1d"].isna().sum()} (expected 20)')
print(f'  market_return_1d min  : {df["market_return_1d"].min():.4f}')
print(f'  market_return_1d max  : {df["market_return_1d"].max():.4f}')
print(f'  market_return_1d mean : {df["market_return_1d"].mean():.4f}')
print()

# Step 3 - show the 5 worst market days in the dataset
print('5 Worst Market Days (all 20 stocks averaged):')
worst = market_daily.nsmallest(5).reset_index()
worst.columns = ['Date', 'avg_return']
worst['avg_return_pct'] = (worst['avg_return'] * 100).round(2)
print(worst.to_string(index=False))

print()
print('5 Best Market Days (all 20 stocks averaged):')
best = market_daily.nlargest(5).reset_index()
best.columns = ['Date', 'avg_return']
best['avg_return_pct'] = (best['avg_return'] * 100).round(2)
print(best.to_string(index=False))

Market-Wide Feature computed
  market_return_1d NaNs : 20 (expected 20)
  market_return_1d min  : -0.1273
  market_return_1d max  : 0.1104
  market_return_1d mean : 0.0011

5 Worst Market Days (all 20 stocks averaged):
      Date  avg_return  avg_return_pct
2020-03-16   -0.127325          -12.73
2020-03-12   -0.093315           -9.33
2020-03-09   -0.072829           -7.28
2020-06-11   -0.055991           -5.60
2022-09-13   -0.048781           -4.88

5 Best Market Days (all 20 stocks averaged):
      Date  avg_return  avg_return_pct
2020-03-13    0.110387           11.04
2020-03-24    0.092742            9.27
2020-04-06    0.073673            7.37
2022-11-10    0.065220            6.52
2020-03-17    0.061705            6.17


In [18]:
# ============================================================
# CELL 16 - Step 4 Validation
# ============================================================
# Verify the feature is correctly mapped to all 20 tickers
# On any given date all 20 tickers must share the same value
# If they differ the map went wrong
# ============================================================

# Pick one random date and confirm all 20 tickers have same value
sample_date = pd.Timestamp('2020-03-16')
sample = df[df['Date'] == sample_date][['Ticker', 'return_1d', 'market_return_1d']]
print(f'Verification — all tickers on {sample_date.date()}')
print('market_return_1d must be identical across all rows')
print('=' * 50)
print(sample.to_string(index=False))
print()
print(f'Unique market_return_1d values on this date : '
      f'{sample["market_return_1d"].nunique()} (must be 1)')
print()
print(f'Total features so far : 6 price + 6 technical + 3 volume + 1 market = 16')
print(f'Dataframe shape       : {df.shape}')

Verification — all tickers on 2020-03-16
market_return_1d must be identical across all rows
Ticker  return_1d  market_return_1d
  AAPL  -0.128647         -0.127325
  ADBE  -0.147452         -0.127325
  AMZN  -0.053698         -0.127325
   BAC  -0.153974         -0.127325
   CRM  -0.158885         -0.127325
 GOOGL  -0.116341         -0.127325
    GS  -0.127053         -0.127325
   JNJ  -0.053317         -0.127325
   JPM  -0.149649         -0.127325
    MA  -0.127254         -0.127325
  META  -0.142530         -0.127325
  MSFT  -0.147390         -0.127325
  NFLX  -0.111389         -0.127325
  NVDA  -0.184521         -0.127325
  ORCL  -0.108700         -0.127325
   PFE  -0.077346         -0.127325
  TSLA  -0.185778         -0.127325
   UNH  -0.172769         -0.127325
     V  -0.135472         -0.127325
   WMT  -0.064330         -0.127325

Unique market_return_1d values on this date : 1 (must be 1)

Total features so far : 6 price + 6 technical + 3 volume + 1 market = 16
Dataframe shape  

-----------------------------------------------

## Step 5 — Time-Based Features

Markets are not random across time — they have seasonal patterns.
These are well documented in financial research:

Monday Effect   : Mondays tend to have lower returns than other days
                  Bad news is often released over weekends
                  Investors react negatively on Monday open

January Effect  : Stocks tend to rise in January
                  Fund managers rebalance portfolios at year start
                  Tax loss selling in December creates January bounce

Earnings Season : Q1 results in April, Q2 in July,
                  Q3 in October, Q4 in January
                  Volatility increases around these periods

These are not guaranteed patterns — but they are real statistical tendencies
that have persisted for decades. The model can learn to weight them.

Features built here:
  day_of_week  →  0=Monday  1=Tuesday  2=Wednesday  3=Thursday  4=Friday
  month        →  1 to 12
  quarter      →  1 to 4

No groupby needed — Date column is the same for all tickers on a given day.
No rolling windows — no NaNs created by this step.

In [19]:
# ============================================================
# CELL 17 - Time-Based Features
# ============================================================
# Extract calendar information directly from the Date column
# dayofweek : 0=Monday to 4=Friday (markets closed weekends)
# month     : 1 to 12
# quarter   : 1 to 4
# No groupby needed - simple datetime extraction
# No NaNs created - every row has a valid Date
# ============================================================

df['day_of_week'] = df['Date'].dt.dayofweek
df['month']       = df['Date'].dt.month
df['quarter']     = df['Date'].dt.quarter

print('Time-Based Features computed')
print('=' * 45)
print(f'  day_of_week NaNs : {df["day_of_week"].isna().sum()} (expected 0)')
print(f'  month NaNs       : {df["month"].isna().sum()} (expected 0)')
print(f'  quarter NaNs     : {df["quarter"].isna().sum()} (expected 0)')
print()
print(f'  day_of_week unique values : {sorted(df["day_of_week"].unique())}')
print(f'  month unique values       : {sorted(df["month"].unique())}')
print(f'  quarter unique values     : {sorted(df["quarter"].unique())}')

Time-Based Features computed
  day_of_week NaNs : 0 (expected 0)
  month NaNs       : 0 (expected 0)
  quarter NaNs     : 0 (expected 0)

  day_of_week unique values : [np.int32(0), np.int32(1), np.int32(2), np.int32(3), np.int32(4)]
  month unique values       : [np.int32(1), np.int32(2), np.int32(3), np.int32(4), np.int32(5), np.int32(6), np.int32(7), np.int32(8), np.int32(9), np.int32(10), np.int32(11), np.int32(12)]
  quarter unique values     : [np.int32(1), np.int32(2), np.int32(3), np.int32(4)]


In [20]:
# ============================================================
# CELL 18 - Step 5 Validation
# ============================================================
# Confirm time features look correct with a sample
# Check day distribution - should be roughly equal across 5 days
# Check month distribution - should be roughly equal across 12 months
# ============================================================

print('Step 5 Complete - Time Feature Validation')
print('=' * 45)

# Day of week distribution
print('Day of week distribution (0=Mon, 4=Fri):')
day_counts = df['day_of_week'].value_counts().sort_index()
day_names  = {0:'Mon', 1:'Tue', 2:'Wed', 3:'Thu', 4:'Fri'}
for day, count in day_counts.items():
    print(f'  {day_names[day]} ({day}) : {count:,} rows')

print()

# Quarter distribution
print('Quarter distribution:')
qtr_counts = df['quarter'].value_counts().sort_index()
for qtr, count in qtr_counts.items():
    print(f'  Q{qtr} : {count:,} rows')

print()
print(f'Total features so far : 6 price + 6 technical + 3 volume + 1 market + 3 time = 19')
print(f'Dataframe shape       : {df.shape}')

Step 5 Complete - Time Feature Validation
Day of week distribution (0=Mon, 4=Fri):
  Mon (0) : 5,620 rows
  Tue (1) : 6,220 rows
  Wed (2) : 6,180 rows
  Thu (3) : 6,100 rows
  Fri (4) : 6,060 rows

Quarter distribution:
  Q1 : 7,380 rows
  Q2 : 7,520 rows
  Q3 : 7,660 rows
  Q4 : 7,620 rows

Total features so far : 6 price + 6 technical + 3 volume + 1 market + 3 time = 19
Dataframe shape       : (30180, 26)


-------------------------------------------------------------------

## Step 6 — Target Variable

This is the most critical step in the entire notebook.
Everything built so far is INPUT to the model.
The target variable is what the model is trying to PREDICT.

Getting this wrong silently destroys the model — no error will warn you.

### What we predict
Will this stock's Close price be meaningfully higher or lower
5 trading days from today?

### Why 5 days
1 day  →  too noisy, random walk dominates, no model can predict reliably
5 days →  one trading week, smooths noise, still actionable as a signal
20 days → too far ahead, market conditions change completely

### How we compute it
  future_return = (Close 5 days later / Close today) - 1

  If future_return >  0.005  →  2  (Buy)
  If future_return < -0.005  →  0  (Sell)
  Otherwise                  →  1  (Hold)

### Why ±0.5% threshold
  A 0.1% move is noise — transaction costs alone exceed this
  ±0.5% filters noise while keeping enough Buy and Sell labels
  Result is a balanced three-class problem the model can learn

### The leakage warning
  shift(-5) looks 5 rows INTO THE FUTURE
  This means the last 5 rows per ticker have NO valid target
  They must become NaN and be dropped in Step 7
  If we forget this — the model trains on corrupted future data

# Understanding `shift(-5)` for Target Variable

---

## The Core Idea

You are thinking about it correctly from a **human perspective**:

> *"I want to look 5 days into the future and compare that price to today."*

But `shift()` works from the **data's perspective** —  
it physically moves the column **up or down** inside the dataframe.

---

## Step 1 — Start With Raw Close Prices

Imagine AAPL's Close column sitting in the dataframe like this:

| Row | Date   | Close |
|-----|--------|-------|
| 1   | Jan 02 | $150  |
| 2   | Jan 03 | $152  |
| 3   | Jan 06 | $148  |
| 4   | Jan 07 | $155  |
| 5   | Jan 08 | $160  |
| 6   | Jan 09 | $162  |
| 7   | Jan 10 | $158  |

Row 1 is today (Jan 02). Row 5 is 5 trading days later (Jan 08).  
We want to compare `$160` (future) to `$150` (today).

---

## Step 2 — What `shift(-5)` Does

It **pulls the entire column UP by 5 positions** —  
bringing future values to sit next to their corresponding current row.

| Row | Date   | Close | `Close.shift(-5)` | What happened?                        |
|-----|--------|-------|-------------------|---------------------------------------|
| 1   | Jan 02 | $150  | **$160**          | ← Jan 08 price pulled back to Jan 02 |
| 2   | Jan 03 | $152  | **$162**          | ← Jan 09 price pulled back to Jan 03 |
| 3   | Jan 06 | $148  | **$158**          | ← Jan 10 price pulled back to Jan 06 |
| 4   | Jan 07 | $155  | NaN               | ← nothing 5 rows ahead               |
| 5   | Jan 08 | $160  | NaN               | ← nothing 5 rows ahead               |

Now on **Row 1 (Jan 02)** we have both values in the same row:
- `Close` = $150 → today's price
- `Close.shift(-5)` = $160 → price 5 days from now

---

## Step 3 — Compute Future Return

```python
future_return = Close.shift(-5) / Close - 1
             = $160 / $150 - 1
             = +0.0667
             = +6.67%  →  BUY ✅
```

No need to loop, no need to look ahead manually.  
The future price is already sitting in the same row — ready to divide.

---

## The Simple Mental Model

```
shift(+5)  →  push column DOWN  →  brings PAST values to current row
shift(-5)  →  pull column UP    →  brings FUTURE values to current row
```

> `shift(-5)` does not move YOU forward 5 days.  
> It moves the **FUTURE DATA backwards** to sit next to you today.

---

## Step 4 — Apply the ±0.5% Threshold

Once we have `future_return` on every row, we classify it:

```
future_return >  +0.005  →  2  (Buy)   stock will be meaningfully higher
future_return <  -0.005  →  0  (Sell)  stock will be meaningfully lower
between the two          →  1  (Hold)  move is too small to act on
```

| Row | Date   | Close | future_return | target |
|-----|--------|-------|---------------|--------|
| 1   | Jan 02 | $150  | +6.67%        | 2 (Buy)  |
| 2   | Jan 03 | $152  | +0.003%       | 1 (Hold) |
| 3   | Jan 06 | $148  | -2.10%        | 0 (Sell) |

---

## Step 5 — The Last 5 Rows Are Always NaN

Because `shift(-5)` pulls rows up, the last 5 rows per ticker  
have **nothing to pull from** — no future data exists yet.

| Row  | Date   | Close | `Close.shift(-5)` | target |
|------|--------|-------|-------------------|--------|
| 1505 | Dec 24 | $250  | NaN               | NaN ❌ |
| 1506 | Dec 26 | $252  | NaN               | NaN ❌ |
| 1507 | Dec 27 | $248  | NaN               | NaN ❌ |
| 1508 | Dec 29 | $251  | NaN               | NaN ❌ |
| 1509 | Dec 30 | $255  | NaN               | NaN ❌ |

These rows **must be dropped** before training.  
5 rows × 20 tickers = **100 rows dropped** in Step 7.

> ⚠️ If these NaN rows were filled with any value, the model  
> would train on fake targets — silently corrupting everything.

---

## Full Picture in One Diagram

```
AAPL Close Column (simplified)

  TODAY          FUTURE (5 days ahead)
  ─────          ─────────────────────
  Jan 02 $150 ──────────────────────────→ Jan 08 $160
                                                  │
                      shift(-5) pulls this ───────┘
                      back to sit on Jan 02

  Result on Jan 02 row:
  ┌──────────┬───────┬────────────────┬───────────────┬────────┐
  │ Date     │ Close │ Close.shift(-5)│ future_return │ target │
  ├──────────┼───────┼────────────────┼───────────────┼────────┤
  │ Jan 02   │ $150  │ $160           │ +6.67%        │ 2(Buy) │
  └──────────┴───────┴────────────────┴───────────────┴────────┘
```


In [21]:
# ============================================================
# CELL 19 - Future Return Computation
# ============================================================
# future_return = Close 5 days ahead / Close today - 1
# shift(-5) moves the future Close back to the current row
# groupby ensures AAPL's future Close never touches MSFT
# Negative shift = look forward in time
# The last 5 rows per ticker will be NaN — this is correct
# ============================================================

df['future_return'] = df.groupby('Ticker')['Close'].transform(
    lambda x: x.shift(-5) / x - 1
)

print('Future Return computed')
print('=' * 45)
print(f'  future_return NaNs : {df["future_return"].isna().sum()} (expected {5*20})')
print(f'  future_return min  : {df["future_return"].min():.4f}')
print(f'  future_return max  : {df["future_return"].max():.4f}')
print(f'  future_return mean : {df["future_return"].mean():.4f}')
print()

# Verify last 5 rows of one ticker are NaN
print('Last 7 rows of AAPL (last 5 must be NaN):')
aapl = df[df['Ticker'] == 'AAPL'][['Date', 'Close', 'future_return']].tail(7)
print(aapl.to_string(index=False))

Future Return computed
  future_return NaNs : 100 (expected 100)
  future_return min  : -0.4309
  future_return max  : 0.5648
  future_return mean : 0.0054

Last 7 rows of AAPL (last 5 must be NaN):
      Date      Close  future_return
2024-12-19 248.204193       0.023220
2024-12-20 252.874390      -0.008999
2024-12-23 253.649414            NaN
2024-12-24 256.560852            NaN
2024-12-26 257.375549            NaN
2024-12-27 253.967392            NaN
2024-12-30 250.598892            NaN


In [22]:
# ============================================================
# CELL 20 - Apply Threshold and Create Target (CORRECTED)
# ============================================================
# Widening threshold from ±0.5% to ±2.0%
# Reason : ±0.5% produced only 11.8% Hold rows
# ±2.0% gives a balanced three class distribution
# A 2% move over 5 days is a meaningful, actionable signal
# ============================================================

conditions = [
    df['future_return'] >  0.02,
    df['future_return'] < -0.02
]
choices = [2, 0]

df['target'] = np.select(conditions, choices, default=1)

# Restore NaN where future_return is NaN
df.loc[df['future_return'].isna(), 'target'] = np.nan

print('Target variable created (±2% threshold)')
print('=' * 45)
print(f'  target NaNs : {df["target"].isna().sum()} (expected 100)')
print()
print('Class distribution:')
counts = df['target'].value_counts().sort_index()
total  = counts.sum()
labels = {0.0: 'Sell', 1.0: 'Hold', 2.0: 'Buy'}
for label, count in counts.items():
    pct = count / total * 100
    bar = '█' * int(pct / 2)
    print(f'  {labels[label]} ({int(label)}) : {count:>6,}  ({pct:.1f}%)  {bar}')

print()
print('Per Ticker distribution:')
print('=' * 55)
print(f'{"Ticker":<8} {"Sell(0)":>8} {"Hold(1)":>8} {"Buy(2)":>8}')
print('-' * 55)
for ticker in sorted(df['Ticker'].unique()):
    tdf  = df[df['Ticker'] == ticker]['target'].dropna()
    sell = int((tdf == 0).sum())
    hold = int((tdf == 1).sum())
    buy  = int((tdf == 2).sum())
    print(f'{ticker:<8} {sell:>8,} {hold:>8,} {buy:>8,}')
print('=' * 55)

Target variable created (±2% threshold)
  target NaNs : 100 (expected 100)

Class distribution:
  Sell (0) :  7,165  (23.8%)  ███████████
  Hold (1) : 13,039  (43.3%)  █████████████████████
  Buy (2) :  9,876  (32.8%)  ████████████████

Per Ticker distribution:
Ticker    Sell(0)  Hold(1)   Buy(2)
-------------------------------------------------------
AAPL          322      625      557
ADBE          398      577      529
AMZN          383      614      507
BAC           396      597      511
CRM           412      575      517
GOOGL         356      636      512
GS            365      633      506
JNJ           261      944      299
JPM           316      717      471
MA            291      763      450
META          410      520      574
MSFT          316      685      503
NFLX          415      551      538
NVDA          441      360      703
ORCL          331      709      464
PFE           397      705      402
TSLA          538      310      656
UNH           331      751      42

In [23]:
# ============================================================
# CELL 21 - Step 6 Validation
# ============================================================
# Check class balance per ticker — imbalance in one stock
# could indicate a data issue specific to that ticker
# All tickers should show roughly similar Buy/Sell/Hold splits
# ============================================================

print('Step 6 Complete - Class Distribution Per Ticker')
print('=' * 55)
print(f'{"Ticker":<8} {"Sell(0)":>8} {"Hold(1)":>8} {"Buy(2)":>8} {"Total":>8}')
print('-' * 55)

for ticker in sorted(df['Ticker'].unique()):
    tdf    = df[df['Ticker'] == ticker]['target'].dropna()
    sell   = int((tdf == 0).sum())
    hold   = int((tdf == 1).sum())
    buy    = int((tdf == 2).sum())
    total  = sell + hold + buy
    print(f'{ticker:<8} {sell:>8,} {hold:>8,} {buy:>8,} {total:>8,}')

print('=' * 55)
print(f'\nDataframe shape : {df.shape}')
print(f'Columns         : {list(df.columns)}')

Step 6 Complete - Class Distribution Per Ticker
Ticker    Sell(0)  Hold(1)   Buy(2)    Total
-------------------------------------------------------
AAPL          322      625      557    1,504
ADBE          398      577      529    1,504
AMZN          383      614      507    1,504
BAC           396      597      511    1,504
CRM           412      575      517    1,504
GOOGL         356      636      512    1,504
GS            365      633      506    1,504
JNJ           261      944      299    1,504
JPM           316      717      471    1,504
MA            291      763      450    1,504
META          410      520      574    1,504
MSFT          316      685      503    1,504
NFLX          415      551      538    1,504
NVDA          441      360      703    1,504
ORCL          331      709      464    1,504
PFE           397      705      402    1,504
TSLA          538      310      656    1,504
UNH           331      751      422    1,504
V             277      835      392    1,

----------------------

## Step 7 — Drop NaNs and Final Save

This is the final step. We have three things to do:

1. Drop future_return column
   It was only needed to compute the target.
   It must not be a feature — it contains future data.
   Leaving it in would cause catastrophic data leakage.
   The model would learn to read the answer directly.

2. Drop all NaN rows
   Rolling windows created NaNs at the start of each ticker.
   shift(-5) created NaNs at the end of each ticker.
   A single dropna() removes all of them in one shot.
   The controlling filter is MA50 — 49 rows per ticker.

3. Save to ../data/processed/all_stocks_features.csv
   This is the final clean feature set.
   Notebook 04 (model training) will load exactly this file.
   We never modify raw data — this is a separate processed file.

Expected rows after dropna:
  30,180 total rows
  - 980 NaN rows from MA50 (49 rows × 20 tickers)
  - 100 NaN rows from target (5 rows × 20 tickers)
  Some rows overlap so actual drop is slightly less than 1,080
  Expected final : ~29,100 to 29,200 rows

In [24]:
# ============================================================
# CELL 22 - Drop future_return Column
# ============================================================
# future_return is intermediate — used only to build target
# It contains future price data — must never be a model feature
# Leakage check : after dropping, no column should contain
# any information about prices beyond today's date
# ============================================================

df = df.drop(columns=['future_return'])

print('future_return column dropped')
print('=' * 45)
print(f'  Columns remaining : {df.shape[1]}')
print(f'  Shape             : {df.shape}')
print()
print('Remaining columns:')
for i, col in enumerate(df.columns, 1):
    print(f'  {i:>2}. {col}')

future_return column dropped
  Columns remaining : 27
  Shape             : (30180, 27)

Remaining columns:
   1. Date
   2. Close
   3. High
   4. Low
   5. Open
   6. Volume
   7. Ticker
   8. return_1d
   9. return_5d
  10. return_10d
  11. return_20d
  12. price_vs_ma20
  13. price_vs_ma50
  14. rsi_14
  15. macd
  16. macd_signal
  17. macd_hist
  18. bb_width
  19. atr_14
  20. volume_ratio
  21. volume_trend
  22. price_volume_signal
  23. market_return_1d
  24. day_of_week
  25. month
  26. quarter
  27. target


In [25]:
# ============================================================
# CELL 23 - Drop NaN Rows
# ============================================================
# Single dropna() removes all NaN rows across all columns
# The MA50 warmup (49 rows per ticker) is the biggest filter
# Target NaNs (5 rows per ticker) also removed here
# We verify the drop makes sense per ticker
# ============================================================

rows_before = len(df)
df = df.dropna().reset_index(drop=True)
rows_after  = len(df)
rows_dropped = rows_before - rows_after

print('NaN rows dropped')
print('=' * 45)
print(f'  Rows before : {rows_before:,}')
print(f'  Rows dropped: {rows_dropped:,}')
print(f'  Rows after  : {rows_after:,}')
print()

# Verify per ticker row count — should all be equal
print('Rows per ticker after dropna:')
ticker_counts = df.groupby('Ticker').size()
print(ticker_counts.to_string())
print()
print(f'  Min rows per ticker : {ticker_counts.min()}')
print(f'  Max rows per ticker : {ticker_counts.max()}')
print(f'  All equal           : {ticker_counts.nunique() == 1}')

NaN rows dropped
  Rows before : 30,180
  Rows dropped: 1,080
  Rows after  : 29,100

Rows per ticker after dropna:
Ticker
AAPL     1455
ADBE     1455
AMZN     1455
BAC      1455
CRM      1455
GOOGL    1455
GS       1455
JNJ      1455
JPM      1455
MA       1455
META     1455
MSFT     1455
NFLX     1455
NVDA     1455
ORCL     1455
PFE      1455
TSLA     1455
UNH      1455
V        1455
WMT      1455

  Min rows per ticker : 1455
  Max rows per ticker : 1455
  All equal           : True


In [26]:
# ============================================================
# CELL 24 - Final Validation Before Save
# ============================================================
# Full sanity check on the final feature set
# Zero NaNs anywhere — non negotiable before saving
# Target class balance one final time on clean data
# Feature summary statistics
# ============================================================

print('FINAL DATASET VALIDATION')
print('=' * 55)
print(f'  Shape          : {df.shape}')
print(f'  Total NaNs     : {df.isnull().sum().sum()} (must be 0)')
print(f'  Tickers        : {df["Ticker"].nunique()}')
print(f'  Date range     : {df["Date"].min().date()} to {df["Date"].max().date()}')
print()

# Feature columns only (exclude original OHLCV and target)
feature_cols = [
    'return_1d', 'return_5d', 'return_10d', 'return_20d',
    'price_vs_ma20', 'price_vs_ma50',
    'rsi_14', 'macd', 'macd_signal', 'macd_hist',
    'bb_width', 'atr_14',
    'volume_ratio', 'volume_trend', 'price_volume_signal',
    'market_return_1d',
    'day_of_week', 'month', 'quarter'
]

print(f'  Feature count  : {len(feature_cols)}')
print()

# Final class distribution on clean data
print('Final Target Distribution:')
counts = df['target'].value_counts().sort_index()
total  = counts.sum()
labels = {0.0: 'Sell', 1.0: 'Hold', 2.0: 'Buy'}
for label, count in counts.items():
    pct = count / total * 100
    bar = '█' * int(pct / 2)
    print(f'  {labels[label]} ({int(label)}) : {count:>6,}  ({pct:.1f}%)  {bar}')

print()
print('NaN check per column:')
nan_counts = df[feature_cols + ['target']].isnull().sum()
if nan_counts.sum() == 0:
    print('  All 20 features + target : 0 NaNs ✅')
else:
    print(nan_counts[nan_counts > 0])

FINAL DATASET VALIDATION
  Shape          : (29100, 27)
  Total NaNs     : 0 (must be 0)
  Tickers        : 20
  Date range     : 2019-03-14 to 2024-12-20

  Feature count  : 19

Final Target Distribution:
  Sell (0) :  7,043  (24.2%)  ████████████
  Hold (1) : 12,530  (43.1%)  █████████████████████
  Buy (2) :  9,527  (32.7%)  ████████████████

NaN check per column:
  All 20 features + target : 0 NaNs ✅


In [27]:
# ============================================================
# CELL 25 - Save Final Feature Set
# ============================================================
# Save to ../data/processed/all_stocks_features.csv
# This is the file notebook 04 will load for model training
# index=False — we do not want pandas row numbers in the CSV
# Print final confirmation with full path
# ============================================================

SAVE_PATH = '../data/processed/all_stocks_features.csv'

df.to_csv(SAVE_PATH, index=False)

# Verify file was saved correctly
saved_df = pd.read_csv(SAVE_PATH)

print('FEATURE ENGINEERING COMPLETE')
print('=' * 55)
print(f'  Saved to       : {SAVE_PATH}')
print(f'  Rows saved     : {len(saved_df):,}')
print(f'  Columns saved  : {saved_df.shape[1]}')
print(f'  File size      : {os.path.getsize(SAVE_PATH) / 1024 / 1024:.1f} MB')
print()
print('=' * 55)
print('Next step : 04_model_training.ipynb')
print('Load file : ../data/processed/all_stocks_features.csv')
print('=' * 55)

FEATURE ENGINEERING COMPLETE
  Saved to       : ../data/processed/all_stocks_features.csv
  Rows saved     : 29,100
  Columns saved  : 27
  File size      : 12.0 MB

Next step : 04_model_training.ipynb
Load file : ../data/processed/all_stocks_features.csv
